# 06 — Jump Diffusion Pricing

Merton's (1976) jump-diffusion model extends GBM with Poisson-driven jumps:

$$dS_t = \mu\,S_t\,dt + \sigma\,S_t\,dW_t + S_t\,(e^{J_t} - 1)\,dN_t$$

where $N_t$ is a Poisson process with intensity $\lambda$, and $J_t \sim
\mathcal{N}(\mu_J, \delta^2)$ is the log-jump size.

This notebook shows how to:

1. Set up a `JDProcess` with jump parameters
2. Price European options under jump diffusion (MC only)
3. Compare GBM vs jump diffusion prices
4. Explore the effect of jump parameters

## 1) Setup

In [1]:
import datetime as dt

import derivatives_pricing as dp

In [2]:
pricing_date = dt.datetime(2025, 6, 15)
maturity = dt.datetime(2026, 6, 15)
r = 0.05

curve = dp.DiscountCurve.flat(rate=r, end_time=2.0)
market = dp.MarketData(pricing_date=pricing_date, discount_curve=curve, currency="USD")

sim_config = dp.SimulationConfig(paths=100_000, end_date=maturity, num_steps=100)
mc_params = dp.MonteCarloParams(random_seed=42)

spec = dp.VanillaSpec(
    option_type=dp.OptionType.CALL,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

## 2) GBM and BSM Baselines

In [3]:
gbm = dp.GBMProcess(
    market_data=market,
    process_params=dp.GBMParams(initial_value=100.0, volatility=0.20),
    sim_config=sim_config,
)

mc_gbm = dp.OptionValuation(
    underlying=gbm, spec=spec, pricing_method=dp.PricingMethod.MONTE_CARLO, params=mc_params
)

# BSM analytical reference
underlying = dp.UnderlyingData(initial_value=100.0, volatility=0.20, market_data=market)
bsm = dp.OptionValuation(underlying=underlying, spec=spec, pricing_method=dp.PricingMethod.BSM)

print(f"BSM analytical: {bsm.present_value():.4f}")
print(f"GBM Monte Carlo: {mc_gbm.present_value():.4f}")

BSM analytical: 10.4506
GBM Monte Carlo: 10.5185


## 3) Jump Diffusion Pricing

The `JDProcess` extends GBM with three additional parameters:

- `lambd` — jump intensity (expected number of jumps per year)
- `mu` — mean of the log-jump size
- `delta` — standard deviation of the log-jump size

In [4]:
jd = dp.JDProcess(
    market_data=market,
    process_params=dp.JDParams(
        initial_value=100.0,
        volatility=0.20,
        lambd=1.0,  # ~1 jump per year
        mu=-0.05,  # average downward jump of ~5%
        delta=0.10,  # jump size std dev of 10%
    ),
    sim_config=sim_config,
)

mc_jd = dp.OptionValuation(
    underlying=jd, spec=spec, pricing_method=dp.PricingMethod.MONTE_CARLO, params=mc_params
)
print(f"Jump diffusion call: {mc_jd.present_value():.4f}")
print(f"GBM call (MC):       {mc_gbm.present_value():.4f}")
print(f"BSM analytical:      {bsm.present_value():.4f}")

Jump diffusion call: 11.4237
GBM call (MC):       10.5185
BSM analytical:      10.4506


## 4) Impact on Puts

With negative mean jump size ($\mu_J < 0$), jump diffusion increases put
prices because crashes become more likely.

In [5]:
put_spec = dp.VanillaSpec(
    option_type=dp.OptionType.PUT,
    exercise_type=dp.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

mc_gbm_put = dp.OptionValuation(
    underlying=gbm, spec=put_spec, pricing_method=dp.PricingMethod.MONTE_CARLO, params=mc_params
)
mc_jd_put = dp.OptionValuation(
    underlying=jd, spec=put_spec, pricing_method=dp.PricingMethod.MONTE_CARLO, params=mc_params
)
print(f"GBM put (MC):            {mc_gbm_put.present_value():.4f}")
print(f"Jump diffusion put (MC): {mc_jd_put.present_value():.4f}")

GBM put (MC):            5.6101
Jump diffusion put (MC): 6.5644


## 5) Varying Jump Intensity

As jump intensity increases, the model departs further from GBM.
At $\lambda = 0$, JD reduces to pure GBM — any small difference from the
GBM row is MC sampling noise.

In [6]:
bsm_ref = bsm.present_value()
print(f"{'Lambda':<10} {'JD Call':>10} {'BSM':>10}")
print("-" * 32)
for lambd in [0.0, 0.5, 1.0, 2.0, 5.0]:
    jd_var = dp.JDProcess(
        market_data=market,
        process_params=dp.JDParams(
            initial_value=100.0, volatility=0.20, lambd=lambd, mu=-0.05, delta=0.10
        ),
        sim_config=sim_config,
    )
    val = dp.OptionValuation(
        underlying=jd_var,
        spec=spec,
        pricing_method=dp.PricingMethod.MONTE_CARLO,
        params=mc_params,
    )
    print(f"{lambd:<10.1f} {val.present_value():>10.4f} {bsm_ref:>10.4f}")

Lambda        JD Call        BSM
--------------------------------


0.0           10.4093    10.4506
0.5           10.9775    10.4506
1.0           11.4237    10.4506
2.0           12.3567    10.4506
5.0           14.8042    10.4506


## 6) OTM Puts — Crash Risk

Jump diffusion has the biggest impact on far OTM puts, which are sensitive
to tail events. The percentage difference grows as puts move further OTM.

In [7]:
print(f"{'Strike':<10} {'GBM Put':>10} {'JD Put':>10} {'Diff %':>10}")
print("-" * 42)
for K in [100, 95, 90, 85, 80]:
    otm_put = dp.VanillaSpec(
        option_type=dp.OptionType.PUT,
        exercise_type=dp.ExerciseType.EUROPEAN,
        strike=float(K),
        maturity=maturity,
    )
    gbm_pv = dp.OptionValuation(
        underlying=gbm, spec=otm_put, pricing_method=dp.PricingMethod.MONTE_CARLO, params=mc_params
    ).present_value()
    jd_pv = dp.OptionValuation(
        underlying=jd, spec=otm_put, pricing_method=dp.PricingMethod.MONTE_CARLO, params=mc_params
    ).present_value()
    pct_diff = (jd_pv / gbm_pv - 1) * 100 if gbm_pv > 0 else float("nan")
    print(f"{K:<10} {gbm_pv:>10.4f} {jd_pv:>10.4f} {pct_diff:>+9.1f}%")

Strike        GBM Put     JD Put     Diff %
------------------------------------------
100            5.6101     6.5644     +17.0%
95             3.7511     4.6344     +23.6%
90             2.3471     3.1190     +32.9%
85             1.3544     1.9849     +46.6%
80             0.7078     1.1858     +67.5%
